# 01 — Exploratory Data Analysis (EDA)

**Dataset:** `creditcard_2023.csv`  
**Goal:** Understand feature distributions, class balance, and correlations before modelling.

In [ ]:
# ── Cell 1: Load CSV, print shape and dtypes ─────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.dpi": 120})

df = pd.read_csv("../data/creditcard_2023.csv")
print(f"Shape: {df.shape}")
print(f"\nColumn dtypes:\n{df.dtypes}")
df.head()

In [ ]:
# ── Cell 2: Check nulls ──────────────────────────────────────────
null_counts = df.isnull().sum()
print("Null counts per column:")
print(null_counts)
print(f"\nTotal nulls in dataset: {null_counts.sum()}")

In [ ]:
# ── Cell 3: Class distribution — value_counts + bar chart ────────
class_counts = df["Class"].value_counts()
print("Class distribution:")
print(class_counts)
print(f"\nFraud ratio: {df['Class'].mean():.4f}")

fig, ax = plt.subplots(figsize=(5, 4))
class_counts.plot.bar(ax=ax, color=["#2ecc71", "#e74c3c"], edgecolor="black")
ax.set_title("Class Distribution", fontsize=14, fontweight="bold")
ax.set_xlabel("Class (0 = Legit, 1 = Fraud)")
ax.set_ylabel("Count")
ax.set_xticklabels(["Legit (0)", "Fraud (1)"], rotation=0)
for i, v in enumerate(class_counts):
    ax.text(i, v + v * 0.01, f"{v:,}", ha="center", fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 4: Amount stats + histogram of Amount vs log1p(Amount) ──
print("Amount statistics:")
print(df["Amount"].describe())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: raw Amount
axes[0].hist(df["Amount"], bins=100, color="#3498db", edgecolor="black", alpha=0.8)
axes[0].set_title("Distribution of Amount", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Amount")
axes[0].set_ylabel("Frequency")

# Right: log1p(Amount)
axes[1].hist(np.log1p(df["Amount"]), bins=100, color="#9b59b6", edgecolor="black", alpha=0.8)
axes[1].set_title("Distribution of log1p(Amount)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("log1p(Amount)")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 5: Correlation heatmap of all features with Class ───────
# Drop id if present
df_corr = df.drop(columns=["id"], errors="ignore")
corr_matrix = df_corr.corr()

fig, ax = plt.subplots(figsize=(16, 12))
sns.heatmap(
    corr_matrix,
    cmap="RdBu_r",
    center=0,
    annot=False,
    linewidths=0.3,
    square=True,
    ax=ax,
)
ax.set_title("Feature Correlation Heatmap", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 6: Top 10 features correlated with Class — bar chart ────
class_corr = (
    corr_matrix["Class"]
    .drop("Class")
    .abs()
    .sort_values(ascending=False)
    .head(10)
)

fig, ax = plt.subplots(figsize=(8, 5))
class_corr.plot.barh(ax=ax, color="#e67e22", edgecolor="black")
ax.set_title("Top 10 Features Correlated with Class", fontsize=14, fontweight="bold")
ax.set_xlabel("|Correlation|")
ax.invert_yaxis()
for i, v in enumerate(class_corr):
    ax.text(v + 0.005, i, f"{v:.3f}", va="center", fontweight="bold")
plt.tight_layout()
plt.show()

## EDA Summary

The dataset contains **568,630 transactions** with 28 PCA-transformed features (V1–V28), a raw `Amount` column, and a binary `Class` label. The classes are **perfectly balanced at 50 / 50**, which is unusual for real-world fraud data and means we can train without heavy class-weight adjustments or resampling. The `Amount` feature is heavily right-skewed, but applying `log1p` produces a much more normally distributed feature — justifying the log-transform added during preprocessing.